In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader

PDF_Path = "telecom_guide.pdf"
loader = PyPDFLoader(PDF_Path)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the pdf")
print("\n ---- page preview (first 500 characters) -----")
print(pages[4].page_content[:500])

Loaded 60 pages from the pdf

 ---- First page preview (first 500 characters) -----
Foreword
 Safeguarding the interests of telecom consumers and empowering them is one of 
the primary objectives of the Telecom Regulatory Authority of India (TRAI). Towards this 
endeavor, TRAI has been issuing Regulations, Directions and orders on various consumer 
centric issues from time to time. To enable consumers and consumer organizations to take 
advantage of these measures, it is important that they are made aware of these initiatives.
 For dissemination of information to consumers, TRA


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,  # ~150 words for chunk
    chunk_overlap=100, # overlap heaps context at bounderies.
    separators=["\n\n", "\n", ".", " "] # tries paragraph - line - sentence - word
)

chunks = splitter.split_documents(pages)
print(len(chunks))

211


In [11]:
print(chunks[0].page_content)

Telecom Regulatory Authority of India
Consumer Handbook on Telecommunications
Mahanagar Doorsanchar Bhavan
Jawaharlal Nehru Marg
New Delhi- 110002
website : www.trai.gov.in


In [12]:
print(chunks[1].page_content)

Foreword
 Safeguarding the interests of telecom consumers and empowering them is one of 
the primary objectives of the Telecom Regulatory Authority of India (TRAI). Towards this 
endeavor, TRAI has been issuing Regulations, Directions and orders on various consumer 
centric issues from time to time. To enable consumers and consumer organizations to take 
advantage of these measures, it is important that they are made aware of these initiatives.
 For dissemination of information to consumers, TRAI follows a multi-pronged approach


In [17]:
print(chunks[2].page_content)
print(chunks[3].page_content)


For dissemination of information to consumers, TRAI follows a multi-pronged approach 
in the form of conduct of consumer outreach programmes, undertaking media campaigns 
and publishing consumer education material. It is with this purpose, a consumer Handbook 
titled ‘Consumer handbook on Telecommunications’ was published by TRAI in February, 
2015.  Telecom is fast moving sector and many new developments have since taken place 
requiring interventions in the form of new Regulations, Tariff orders, Directions and other
requiring interventions in the form of new Regulations, Tariff orders, Directions and other 
initiatives by TRAI. The Handbook has now been revised comprehensively to cover all these 
new developments and interventions. The Handbook is written in a simple consumer friendly 
language for ease of understanding.
 The handbook is intended for free distribution to consumers and the registered 
consumer organizations at the consumer outreach programmes, workshops and seminars 

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-V2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"vector store ready {vector_store._collection.count()} vector stored. ")

D:\tutorial-agentic-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KOTI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-V2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2854.17it/s]


vector store ready 211 vector stored. 


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})
test_query = "What is VoLTE and how does it improves call quality"
retrieved = retriever.invoke(test_query)

for i,doc in enumerate(retrieved,1):
    print(f"--- chunk {1} ---")
    print(doc.page_content[:300])
    print()

--- chunk 1 ---
VAS through a simple process by dialing or 
sending SMS to 155223 (toll free).
Using SMS:
Ø Message/Text the keyword “STOP” from 
the mobile number to 155223.
Ø Receive a reply from 155223 with a 
list of VAS products activated on that 
mobile phone.
For example-
To deactivate, reply with Service Nu

--- chunk 1 ---
users to rate their call after it ends. Caller 
simply selects their rating in the form of stars 
and indicates if the calls were made indoor, 
outdoor or while travelling. Callers can also 
provide additional details such as noise or 
audio delay or mark a call as ‘dropped’. 
Key features:- 
•	 Abi

--- chunk 1 ---
consumer care agent in the IVRS menu. 
Operation of IVRS on Customer Care 
Number
The Interactive Voice Response System 
(IVRS) at the “Customer Care Number” 
operates in the following manner:
Appeal to Appellate Authority
If	 a	 subscriber	 is	 not	 satisfied	 with	 the	
redressal of his complaint,



In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}

"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human","{question}")
])

# --- llm via Groq API ---
llm = ChatGroq(
    model = "qwen/qwen3-32b",
    temperature = 0,
    reasoning_format="parsed"    
)
chain = (
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [23]:
question = "How does intenational roaming work and what charges should I expect?"

print(f"Q: {question} \n")
print("A:", chain.invoke(question))

Q: How does intenational roaming work and what charges should I expect? 

A: The context provided does not contain specific information about international roaming charges or how international roaming works. It only discusses **national roaming charges** and TRAI regulations for India. For details on international roaming, you would need to consult your service provider, as charges vary by country, network, and plan.
